# Splitting and Training Data
## Importing modules

In [ ]:
import pandas as pd
import os 
import mlflow
import mlflow.data
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

## 1. Loading Data 

In [ ]:
df=pd.read_csv('../data/processed/spam_cleaned.csv', encoding='latin-1')

## Setting mlflow 

In [ ]:
ROOT=os.path.abspath("./")
DB_PATH=os.path.join(ROOT, "mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")
mlflow.set_experiment("spam-filter-experiment")

## 2. Splitting Data

In [ ]:
#Splitting the dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(df['text'], df['label'].map({'ham': 0, 'spam': 1}), test_size=0.2, random_state=42, stratify=df['label'])
print(y_test)

## 3.1 Training the Model (Multinomial Naive-bayes)

In [ ]:

# mlflow.set_tag("model", "MultinomialNB")
# mlflow.log_param("dataset", "../Data/Processed/spam_cleaned.csv")
# mlflow.log_param("split_ratio", 0.2)
# mlflow.log_param("random_state", 42)
# mlflow.log_param("train_length", 4135)
# mlflow.log_param("test_length", 1034)
# mnb_model = MultinomialNB()
# mnb_model.fit(x_train, y_train)
# y_pred_mnb = mnb_model.predict(x_test)


## 3.2 Training the Model (Logistic Regression )

In [ ]:
# from sklearn.linear_model import LogisticRegression
# lr_model = LogisticRegression(max_iter=1000, C=100)
# lr_model.fit(x_train, y_train)

## 4. Rating the Models

In [ ]:
# from sklearn.metrics import classification_report, confusion_matrix
# #Evaluating the Multinomial Naive Bayes model
# print("Multinomial Naive Bayes Model \n")
# y_pred_mnb = mnb_model.predict(x_test)
# print(classification_report(y_test, y_pred_mnb))
# print("Confusion Matrix\n", confusion_matrix(y_test, y_pred_mnb))
# #Evaluating the Logistic Regression model
# print("\nLogistic Regression Model \n")
# y_pred_lr = lr_model.predict(x_test)
# print(classification_report(y_test, y_pred_lr))
# print("Confusion Matrix\n", confusion_matrix(y_test, y_pred_lr))

### Cross validation 

In [ ]:
## testing the SVC model with the best hyperparameters to see if the model generalizes data
# from sklearn.model_selection import  cross_validate
# model = SVC(C=1.1644227174200665, kernel='linear', max_iter=2000, random_state=42)
# vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
# lr_scores = cross_validate(model, x_train, y_train, cv=5, return_train_score=True, scoring='f1')
# print("Logistic Regression Validation Scores:", lr_scores['test_score'])
# print("Logistic Regression Training Scores:", lr_scores['train_score'])
# print("mean validation score:", lr_scores['test_score'].mean())
# print("mean training score:", lr_scores['train_score'].mean())
# print("difference between mean training and validation score:", lr_scores['train_score'].mean() - lr_scores['test_score'].mean())

### Hyperparameter Tuning 

In [ ]:
# from sklearn.model_selection import GridSearchCV
# grid = GridSearchCV(lr_model, param_grid={'C': [0.01, 0.1, 1, 10, 100]}, cv=5, scoring='f1')
# grid.fit(x_train, y_train)
# print("Best Parameters for Logistic Regression:", grid.best_params_)
# print("Best F1 Score for Logistic Regression:", grid.best_score_)

### Hyperparameter Tuning using optuna , experiment tracking and model registry

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
def hyperparameter_tuning(model_name):
    def objective(trial):
        if model_name == "Logistic_Regression":
            parameters = {
                'C': trial.suggest_float('C', 0.001, 1000.0, log=True),
                'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'saga'])
            }
            model = LogisticRegression(**parameters)
        elif model_name == "Multinomial_NB":
            parameters = {
                'alpha': trial.suggest_float('alpha', 0.001, 10.0, log=True),
                'fit_prior': trial.suggest_categorical('fit_prior', [True, False]),
                'class_prior': trial.suggest_categorical('class_prior', [None, [0.5, 0.5], [0.7, 0.3], [0.3, 0.7]])

            }
            model = MultinomialNB(**parameters)
        elif model_name == "SVC" :
            parameters = {
                'C': trial.suggest_float('C', 0.001, 1000.0, log=True),
                'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])
            }
            model = SVC(**parameters, max_iter=2000, random_state=42)
        #setting up the pipeline 
        vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
        text_pipeline = Pipeline([('vectorizer', vectorizer), ('classifier', model)])
        #testing for the best hyperparameters using cross-validation
        score = cross_val_score(text_pipeline, x_train, y_train, cv=5, scoring='f1').mean()
        #fitting the pipeline
        text_pipeline.fit(x_train, y_train)
        with mlflow.start_run(run_name=f"Trial_{trial.number}" , nested=True):
            mlflow.log_params(parameters)
            mlflow.log_metric("cv_f1_score", score)
            data_set = mlflow.data.from_pandas(df,source="/home/stayn/Projects/Spam_Filter/data/processed/spam_cleaned.csv" , name="spam_filter_dataset_v1")
            mlflow.log_input(data_set,context="training_testing_data")
            mlflow.sklearn.log_model(text_pipeline, skops_trusted_types=["scipy.sparse._csr.csr_matrix"])
            #final test and evaluation 
            predictions = text_pipeline.predict(x_test)
            eval_df = pd.DataFrame({'test_data': predictions, 'label': y_test})
            mlflow.models.evaluate(data = eval_df, targets='label', predictions='test_data', model_type = "classifier")
        return score
    with mlflow.start_run(run_name=f"{model_name}_models") :
        mlflow.set_tag("model", f"{model_name}")
        study = optuna.create_study(study_name=f"{model_name.lower()}_study", storage=f"sqlite:////home/stayn/Projects/Spam_Filter/notebooks/{model_name.lower()}_v1.db", direction='maximize' , load_if_exists=True)
        study.optimize(objective, n_trials=100 ,show_progress_bar=True)


In [ ]:
algorithms = ["Logistic_Regression", "Multinomial_NB", "SVC"]
for algo in algorithms:
    hyperparameter_tuning(algo)

### Exporting Model and Vectorizer

In [ ]:
# import joblib
# import os
# # Save the trained models and vectorizer
# joblib.dump(lr_model, '../Models/logistic_regression_model.pkl')
# joblib.dump(vectorizer, '../Models/tfidf_vectorizer.pkl')